In [ ]:
!pip install torch transformers pandas scikit-learn tqdm

In [ ]:
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch import nn
from tqdm import tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
MODEL_NAME = "roberta-base"
MAX_LEN = 256
BATCH_SIZE = 8
EPOCHS = 3
LR = 2e-5
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

TRAIN_CSV = "/content/drive/MyDrive/SemEval2026/Subtask1/train_subtask1.csv"
TEST_CSV = "/content/drive/MyDrive/SemEval2026/Subtask1/test_subtask1.csv"
OUTPUT_CSV = "/content/drive/MyDrive/SemEval2026/Subtask1/subtask1_submission.csv"


In [ ]:
class AffectDataset(Dataset):
    def __init__(self, texts, labels=None, tokenizer=None):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            padding="max_length",
            truncation=True,
            max_length=MAX_LEN,
            return_tensors="pt"
        )

        item = {k: v.squeeze(0) for k, v in enc.items()}

        if self.labels is not None:
            item["labels"] = torch.tensor(self.labels[idx], dtype=torch.float)

        return item

In [ ]:
class AffectRegressor(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        self.regressor = nn.Linear(self.encoder.config.hidden_size, 2)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = out.last_hidden_state[:, 0]
        return self.regressor(pooled)


In [ ]:
train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts = train_df["text"].astype(str).tolist()
train_labels = train_df[["valence", "arousal"]].values.tolist()

train_ds = AffectDataset(train_texts, train_labels, tokenizer)
train_dl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)


In [ ]:
model = AffectRegressor(MODEL_NAME).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
loss_fn = nn.MSELoss()

model.train()
for epoch in range(EPOCHS):
    total_loss = 0
    for batch in tqdm(train_dl, desc=f"Epoch {epoch+1}"):
        optimizer.zero_grad()

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        preds = model(input_ids, attention_mask)
        loss = loss_fn(preds, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss: {total_loss / len(train_dl):.4f}")


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 346/346 [02:06<00:00,  2.72it/s]


Epoch 1 Loss: 0.7725


Epoch 2: 100%|██████████| 346/346 [02:06<00:00,  2.74it/s]


Epoch 2 Loss: 0.5939


Epoch 3: 100%|██████████| 346/346 [02:06<00:00,  2.75it/s]

Epoch 3 Loss: 0.5013


In [ ]:
model.eval() #last

test_texts = test_df["text"].astype(str).tolist()
test_ds = AffectDataset(test_texts, labels=None, tokenizer=tokenizer)
test_dl = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

predictions = []

with torch.no_grad():
    for batch in tqdm(test_dl, desc="Predicting"):
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)

        preds = model(input_ids, attention_mask)
        predictions.extend(preds.cpu().numpy())

preds_df = pd.DataFrame(predictions, columns=["pred_valence", "pred_arousal"])

submission = pd.concat(
    [
        test_df[["user_id", "text_id"]].reset_index(drop=True),
        preds_df.reset_index(drop=True)
    ],
    axis=1
)

submission.to_csv(OUTPUT_CSV, index=False)

submission.head()


Predicting: 100%|██████████| 218/218 [00:24<00:00,  9.00it/s]


,user_id,text_id,pred_valence,pred_arousal
0,3,256,0.625531,0.510158
1,3,257,1.081761,0.994829
2,3,258,1.347715,0.758952
3,3,249,1.276936,1.213558
4,3,250,-0.229180,0.124585
